In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import (
    get_model_color, APPROACH_COLORS, MODEL_COLORS
)
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation, bonferroni_correction
)
from ocr_vs_vlm.results_final.shared.viz_utils import (
    setup_plotly_template, create_metric_boxplot, create_significance_heatmap
)
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_dataset_data, extract_models_from_columns
)

setup_plotly_template()

DATASET = 'InfographicVQA_mini'
print(f"✓ Setup complete for {DATASET}")

## 1. Load Data

In [ ]:
try:
    df = load_dataset_data(DATASET, task_type='qa')
    print(f"✓ Loaded {len(df)} samples")
    print(f"  Approaches: {df['approach'].unique().tolist()}")
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

## 2. ANLS by Approach and Model

In [ ]:
# Compute stats
stats = []
for approach in df['approach'].unique() if len(df) > 0 else []:
    app_df = df[df['approach'] == approach]
    for model in models:
        col = f'anls_score_{model}'
        if col in app_df.columns:
            scores = app_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                stats.append({
                    'Approach': approach,
                    'Model': model,
                    'ANLS': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'Std': np.std(scores),
                    'N': len(scores)
                })

stats_df = pd.DataFrame(stats)
if len(stats_df) > 0:
    display(Markdown("### ANLS Summary"))
    pivot = stats_df.pivot_table(index='Model', columns='Approach', values='ANLS').round(4)
    display(pivot)

In [ ]:
# Box plot visualization
if len(stats_df) > 0:
    fig = make_subplots(rows=1, cols=len(df['approach'].unique()),
                        subplot_titles=df['approach'].unique().tolist())
    
    for i, approach in enumerate(df['approach'].unique()):
        app_df = df[df['approach'] == approach]
        for model in models:
            col = f'anls_score_{model}'
            if col in app_df.columns:
                scores = app_df[col].dropna()
                if len(scores) > 0:
                    fig.add_trace(
                        go.Box(y=scores, name=model, 
                              marker_color=get_model_color(model),
                              showlegend=(i == 0)),
                        row=1, col=i+1
                    )
    
    fig.update_layout(
        title=f'{DATASET}: ANLS by Approach and Model',
        height=450,
        boxmode='group'
    )
    fig.show()

## 3. Statistical Significance

In [ ]:
# Pairwise approach comparison
print("📊 Pairwise Comparison (Wilcoxon Signed-Rank)")
print("=" * 60)

approaches = df['approach'].unique().tolist() if len(df) > 0 else []
approach_scores = {}

for approach in approaches:
    app_df = df[df['approach'] == approach]
    all_scores = []
    for model in models:
        col = f'anls_score_{model}'
        if col in app_df.columns:
            all_scores.extend(app_df[col].dropna().tolist())
    approach_scores[approach] = np.array(all_scores)

p_values = []
comparisons = []

for i, a1 in enumerate(approaches):
    for a2 in approaches[i+1:]:
        if len(approach_scores[a1]) > 0 and len(approach_scores[a2]) > 0:
            n = min(len(approach_scores[a1]), len(approach_scores[a2]))
            stat, p = wilcoxon_test(approach_scores[a1][:n], approach_scores[a2][:n])
            d = cohens_d(approach_scores[a1], approach_scores[a2])
            p_values.append(p)
            comparisons.append((a1, a2, p, d))

significant = bonferroni_correction(p_values) if p_values else []

for (a1, a2, p, d), sig in zip(comparisons, significant):
    m1, m2 = np.mean(approach_scores[a1]), np.mean(approach_scores[a2])
    winner = a1 if m1 > m2 else a2
    print(f"\n{a1} vs {a2}:")
    print(f"   Mean: {m1:.4f} vs {m2:.4f} | Winner: {winner}")
    print(f"   p={p:.4f} | d={d:.3f} ({effect_size_interpretation(d)})")
    print(f"   {'✓' if sig else '✗'} Significant (Bonferroni)")

## 4. Error Analysis

In [ ]:
# Find difficult questions (low ANLS across all models)
if len(df) > 0 and 'question' in df.columns:
    df_copy = df.copy()
    anls_cols = [c for c in df.columns if c.startswith('anls_score_')]
    
    if anls_cols:
        df_copy['mean_anls'] = df_copy[anls_cols].mean(axis=1)
        
        display(Markdown("### Hardest Questions (Lowest ANLS)"))
        hard = df_copy.nsmallest(5, 'mean_anls')[['question', 'mean_anls']].round(4)
        display(hard)
        
        display(Markdown("### Easiest Questions (Highest ANLS)"))
        easy = df_copy.nlargest(5, 'mean_anls')[['question', 'mean_anls']].round(4)
        display(easy)

## 5. Key Findings

In [ ]:
print("=" * 60)
print(f"{DATASET} KEY FINDINGS")
print("=" * 60)

if len(stats_df) > 0:
    best = stats_df.loc[stats_df['ANLS'].idxmax()]
    print(f"\n🏆 Best Configuration:")
    print(f"   {best['Approach']} + {best['Model']}: ANLS = {best['ANLS']:.4f}")
    
    approach_means = stats_df.groupby('Approach')['ANLS'].mean().sort_values(ascending=False)
    print(f"\n📊 Approach Ranking:")
    for i, (app, score) in enumerate(approach_means.items(), 1):
        print(f"   {i}. {app}: {score:.4f}")

print("\n" + "=" * 60)